In [5]:
import pandas as pd
import numpy as np

# === 1) Load CSV ===
file_path = "Data/1/FVG_OB_2015-2025.csv"
df = pd.read_csv(file_path, sep=";")

# === 2) Column names ===
group_col = "has_open_fvg_in_range"
profit_col = "profit"
win_col = "is_profitable"
rr_col = "rr_ext"

# Check required columns
required_cols = [group_col, profit_col, win_col, rr_col]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in the file: {missing}")

# === 3) Function to calculate group statistics ===
def calc_group_stats(subdf):
    trades = len(subdf)

    wins = subdf[subdf[win_col] == 1]
    losses = subdf[subdf[win_col] == 0]

    gross_profit = wins[profit_col].sum()
    gross_loss = losses[profit_col].sum()  # usually negative

    if gross_loss != 0:
        profit_factor = gross_profit / abs(gross_loss)
    elif gross_profit > 0:
        profit_factor = np.inf
    else:
        profit_factor = np.nan

    return pd.Series({
        "Trades": trades,
        "Wins": len(wins),
        "Losses": len(losses),
        "Win rate %": len(wins) / trades * 100 if trades else np.nan,
        "Sum profit": subdf[profit_col].sum(),
        "Avg profit": subdf[profit_col].mean(),
        "Median profit": subdf[profit_col].median(),
        "Avg win": wins[profit_col].mean() if len(wins) else np.nan,
        "Avg loss": losses[profit_col].mean() if len(losses) else np.nan,
        "Profit factor": profit_factor,
        "Avg rr_ext": subdf[rr_col].mean()
    })

# === 4) Split by has_open_fvg_in_range ===
result = (
    df.groupby(group_col, dropna=False)
      .apply(calc_group_stats)
      .reset_index()
      .sort_values(group_col)
)

# Share of total trades
result.insert(2, "Share %", result["Trades"] / len(df) * 100)

# === 5) Prepare formatted output ===
result_display = result.copy()

# Round float columns
float_cols = [
    "Share %",
    "Win rate %",
    "Sum profit",
    "Avg profit",
    "Median profit",
    "Avg win",
    "Avg loss",
    "Profit factor",
    "Avg rr_ext"
]

for col in float_cols:
    result_display[col] = result_display[col].astype(float).round(3)

# Convert integer columns to int so they do not show as .000000
int_cols = [group_col, "Trades", "Wins", "Losses"]

for col in int_cols:
    if result_display[col].isna().any():
        result_display[col] = result_display[col].astype("Int64")
    else:
        result_display[col] = result_display[col].astype(int)

# === 6) Display result ===
display(
    result_display.style.format({
        group_col: "{:d}",
        "Trades": "{:d}",
        "Wins": "{:d}",
        "Losses": "{:d}",
        "Share %": "{:.2f}",
        "Win rate %": "{:.2f}",
        "Sum profit": "{:,.2f}",
        "Avg profit": "{:,.2f}",
        "Median profit": "{:,.2f}",
        "Avg win": "{:,.2f}",
        "Avg loss": "{:,.2f}",
        "Profit factor": "{:.3f}",
        "Avg rr_ext": "{:.3f}",
    })
)

,has_open_fvg_in_range,Trades,Share %,Wins,Losses,Win rate %,Sum profit,Avg profit,Median profit,Avg win,Avg loss,Profit factor,Avg rr_ext
0,0,164,56.36,96,68,58.54,"5,980.95",36.47,209.07,776.98,"-1,008.96",1.087,1.025
1,1,127,43.64,40,87,31.50,"-10,310.29",-81.18,-999.68,"1,955.30","-1,017.50",0.884,2.499
